# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb huggingface_hub scikit-learn
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import HfApi

TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"

api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=TOKEN)
fact_files = [f for f in all_files if "fact_content_daily_performance" in f and f"month={MONTH}" in f]
next_month_files = [f for f in all_files if "fact_content_daily_performance" in f and "month=2026-04" in f]
dim_content_files = [f for f in all_files if "dim_content" in f.lower() and f.endswith(".parquet")]

FACT = [f"{BASE}/{f}" for f in fact_files]
FACT_NEXT = [f"{BASE}/{f}" for f in next_month_files]
DIM_CONTENT = [f"{BASE}/{f}" for f in dim_content_files]
FACT_STR = "[" + ", ".join(f"'{p}'" for p in FACT) + "]"
FACT_TWO_MONTHS_STR = "[" + ", ".join(f"'{p}'" for p in FACT + FACT_NEXT) + "]"
DIM_CONTENT_STR = "[" + ", ".join(f"'{p}'" for p in DIM_CONTENT) + "]"

# same feature build as w05, rebuilding it here so this notebook stands on its own
df_content = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet({DIM_CONTENT_STR})").df()
labeled = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position,
           LEAD(gsc_impressions, 30) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS impressions_plus30
    FROM read_parquet({FACT_TWO_MONTHS_STR})
""").df()
labeled = labeled[labeled["report_date"] < "2026-04-01"].dropna(subset=["impressions_plus30"])
labeled["future_decline_label"] = (labeled["impressions_plus30"] < labeled["gsc_impressions"]).astype(int)
labeled = labeled.merge(df_content, on="content_hash_id", how="left")
labeled["report_date"] = pd.to_datetime(labeled["report_date"])
labeled["content_created_date"] = pd.to_datetime(labeled["content_created_date"])
labeled["days_since_update"] = (labeled["report_date"] - labeled["content_created_date"]).dt.days

feat = labeled.groupby(["content_hash_id", "client_hash_id"]).agg(
    impressions_prior90=("gsc_impressions", "sum"),
    clicks_prior90=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_since_update=("days_since_update", "max"),
    future_decline_label=("future_decline_label", "max"),
).reset_index()
feat["ctr_prior90"] = feat["clicks_prior90"] / feat["impressions_prior90"].replace(0, np.nan)
feat = feat.dropna()
feature_cols = ["impressions_prior90", "clicks_prior90", "avg_position", "days_since_update", "ctr_prior90"]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #4 — "The Freshness Multiplier" (365+ day content refreshed shows 3.2x
health boost, 57x impression boost)**

My methodology question: is this a real before/after comparison of the SAME pages
over time, or a cross-sectional comparison of two different groups (refreshed vs.
never-refreshed) measured at one point in time? The paper doesn't say explicitly.
If it's cross-sectional, this isn't evidence that refreshing CAUSES the jump \u2014
it's possible the pages that got selected for refresh were already the stronger,
more strategically important ones to begin with (selection bias), and the refresh
correlates with the outcome rather than causing it. The paper itself is careful
about this exact trap elsewhere (it flags the 361+ freshness bucket's 283:1 ratio
as unstable due to n=1), so I'd genuinely want to know: what's the sample size
behind the 71-to-4039 impression jump specifically, and was it a matched
before/after cohort? This is asked in the spirit of the paper's own standard, not
against it \u2014 it already discloses more caveats than most public studies do.

**The Growth Prediction model (ML Appendix, logistic regression, 71% holdout
accuracy, predicting growing vs. declining pages)**

My methodology question: where does the growth/decline label actually come from?
Per the paper's own definitions section, "Trend Direction" is calculated from
30-day-vs-previous-30-day impression change \u2014 a CURRENT-window comparison, not
a genuine future outcome. If the label describes the same window the features are
measured in, "predicting growth" may be closer to describing a pattern already
present in the data than forecasting something that hasn't happened yet. I'd also
ask about the 80/20 split design specifically: is it time-aware (train on earlier
data, test on strictly later data) or a random split? The paper states the split
ratio but not its design, and given "Days Visible" is one of the top positive
coefficients \u2014 while the correlation appendix shows Days Visible correlating
with Content Age at r=0.5 and negatively with Days Since Update at r=-0.3 \u2014 it's
worth asking whether the model is leaning on a proxy for content age/staleness
rather than an independent, actionable lever. This is the same class of question
I had to ask my own ML-08 model about avg_position and days_since_update.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week-5 already used a client-grouped split, but I never actually showed what a
naive random split would've looked like next to it — so here's the honest
before/after: random row split first (the thing most people do by accident),
then the grouped split I actually used.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k=50):
    top_k = np.argsort(-scores)[:k]
    return y_true.iloc[top_k].mean()

# --- BEFORE: naive random split, rows shuffled with no regard for client ---
np.random.seed(0)
shuffled = feat.sample(frac=1, random_state=0)
cut = int(len(shuffled) * 0.8)
train_naive, test_naive = shuffled.iloc[:cut], shuffled.iloc[cut:]

logreg_naive = LogisticRegression(max_iter=1000).fit(train_naive[feature_cols], train_naive["future_decline_label"])
naive_proba = logreg_naive.predict_proba(test_naive[feature_cols])[:, 1]
naive_auc = roc_auc_score(test_naive["future_decline_label"], naive_proba)
naive_p50 = precision_at_k(test_naive["future_decline_label"].reset_index(drop=True), naive_proba)

# --- AFTER: client-grouped split, same as w05 ---
np.random.seed(42)
clients = feat["client_hash_id"].unique()
np.random.shuffle(clients)
split_point = int(len(clients) * 0.8)
train_clients, test_clients = clients[:split_point], clients[split_point:]
train_grouped = feat[feat["client_hash_id"].isin(train_clients)]
test_grouped = feat[feat["client_hash_id"].isin(test_clients)]

logreg_grouped = LogisticRegression(max_iter=1000).fit(train_grouped[feature_cols], train_grouped["future_decline_label"])
grouped_proba = logreg_grouped.predict_proba(test_grouped[feature_cols])[:, 1]
grouped_auc = roc_auc_score(test_grouped["future_decline_label"], grouped_proba)
grouped_p50 = precision_at_k(test_grouped["future_decline_label"].reset_index(drop=True), grouped_proba)

print(f"naive random split  -> AUC {naive_auc:.3f}, precision@50 {naive_p50:.3f}")
print(f"client-grouped split -> AUC {grouped_auc:.3f}, precision@50 {grouped_p50:.3f}")
print("if the grouped number is lower, that's not a worse model -- it's a more honest one,")
print("the random split was probably letting the model recognize clients it already saw")

naive random split  -> AUC 0.774, precision@50 1.000
client-grouped split -> AUC 0.773, precision@50 1.000
if the grouped number is lower, that's not a worse model -- it's a more honest one,
the random split was probably letting the model recognize clients it already saw


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same leakage hunt from ML-04/ML-08, run again on this week's final feature set,
plus one thing I didn't check last time -- whether the two client groups above
actually don't overlap (should be obvious, but "should be" isn't a check).

In [11]:
overlap = set(train_clients) & set(test_clients)
print("client overlap between train and test:", len(overlap), "(should be 0)")

banned = ["health_score", "priority_score", "action_type", "future_decline_label",
          "impressions_plus30", "trend_direction", "trend_pct"]
used = [c for c in feature_cols if c in banned]
print("banned columns actually in my feature list:", used, "(should be empty)")

# quick gut check on correlations -- nothing should be suspiciously close to 1
corrs = feat[feature_cols].corrwith(feat["future_decline_label"]).sort_values(key=abs, ascending=False)
print("\nfeature correlation with the label, sorted by strength:")
print(corrs)
print("\nif anything up there is like 0.9+, that's a red flag, not a good sign -- go check why")

client overlap between train and test: 0 (should be 0)
banned columns actually in my feature list: [] (should be empty)

feature correlation with the label, sorted by strength:
days_since_update      0.111149
avg_position           0.028380
impressions_prior90    0.011933
clicks_prior90        -0.010010
ctr_prior90           -0.009812
dtype: float64

if anything up there is like 0.9+, that's a red flag, not a good sign -- go check why


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Taking my boldest line from w05 and rewriting it honestly.

In [12]:
# recompute the actual RF result here instead of assuming w05's variable exists in this session
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(
    train_grouped[feature_cols], train_grouped["future_decline_label"]
)
rf_proba_check = rf.predict_proba(test_grouped[feature_cols])[:, 1]
rf_p50 = precision_at_k(test_grouped["future_decline_label"].reset_index(drop=True), rf_proba_check)

original_claim = ("Random Forest achieves precision@50 of {:.3f}, proving the model "
                   "can reliably identify declining pages.").format(rf_p50)

rewritten_claim = (
    "On this month's client-grouped test split, the Random Forest model's top-50 "
    "ranked candidates were observed to include a higher share of pages that later "
    "declined, compared to the ML-07 rule baseline on the same split. This is "
    "decision-support for reviewers, not a guarantee about any individual page, "
    "and hasn't been tested outside this one month's data yet."
)

print("ORIGINAL (overclaims):\n", original_claim)
print("\nREWRITTEN (safe language):\n", rewritten_claim)

ORIGINAL (overclaims):
 Random Forest achieves precision@50 of 1.000, proving the model can reliably identify declining pages.

REWRITTEN (safe language):
 On this month's client-grouped test split, the Random Forest model's top-50 ranked candidates were observed to include a higher share of pages that later declined, compared to the ML-07 rule baseline on the same split. This is decision-support for reviewers, not a guarantee about any individual page, and hasn't been tested outside this one month's data yet.


In [13]:
print("test set size:", len(test_grouped))
print("positive rate in test set:", test_grouped["future_decline_label"].mean())
print("50 out of", len(test_grouped), "is", round(100*50/len(test_grouped), 1), "% of the whole test set")

test set size: 32726
positive rate in test set: 0.9818187373953432
50 out of 32726 is 0.2 % of the whole test set


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.